In [1]:
import gzip
import itertools
from collections import defaultdict 

## Load in the palindrome library, and make a mapping of target to sequence

In [36]:
palendromes = {}

with open("/Users/aaronmck/Desktop/code/base_editing/2019_11_27_Maryam_palendromes/synth_constructs_2018_08_28.fasta") as f:
    for line1,line2 in itertools.zip_longest(*[f]*2):
        pal_id =  line1.split('_')[1][0:20]
        pal_seq = line2.upper()
        # if pal_id in palendromes:
        #    print("Warning: collision on " + pal_id)
        palendromes[pal_id] = pal_seq
print(len(palendromes))     

bsmBI = "GAGACG"
bsmBI_RC = "CGTCTC"
cnt = 0
for sequence in list(palendromes.values()):
    # print(sequence)
    if bsmBI in sequence or bsmBI_RC in sequence:
        cnt += 1
        
print(cnt)


314
0


### Now process our sequencing data

first, a function that takes the reference and sequence alignments and extracts the guide/target pairs


In [39]:
### returns a tuple of the extracted guide and target
def process_ref_seq_alignment(ref,sequence,guide_ref_start=20,guide_ref_stop=40,target_ref_start=128,target_ref_stop=154):
    assert len(ref) == len(sequence), "ref %r and sequence %r are not the same size" % (len(ref), len(sequence))
    ref_pos = 0
    guide = ""
    target = ""
    for ref_base,seq_base in zip(ref,sequence):
        if ref_pos >= guide_ref_start and ref_pos < guide_ref_stop:
            guide += seq_base
        elif ref_pos >= target_ref_start and ref_pos < target_ref_stop:
            target += seq_base
        if ref_base != '-':
            ref_pos += 1
    return((guide,target))


##### now for each pair of sequences (over multiple lines) in the aligned file, count outcomes, removing any misaligned ( containing dashes) sequences

In [42]:
ref = ""
sequence = ""
ref_name_set = False
seq_name_set = False

palendromes_to_outcomes = defaultdict(list)

with gzip.open('../data/merged_reads.fastq.aligned.gz', 'rt') as f:
    for line in f:
        if line.startswith('>'):
            if not ref_name_set:
                ref_name_set = True
            elif (not seq_name_set) and ref_name_set:
                seq_name_set = True
            elif seq_name_set and ref_name_set:
                (guide,target) = process_ref_seq_alignment(ref,sequence)
                if sum([1 if x == '-' else 0 for x in sequence]) < 30:
                    palendromes_to_outcomes[guide].append(target)
                ref = ""
                sequence = ""
                ref_name_set = True
                seq_name_set = False
            else:
                print("WE SHOULDNT BE HERE")
        else:
            if ref_name_set and (not seq_name_set):
                ref += line.strip("\n")
            else:
                sequence += line.strip("\n")

#### Now match up each read's target sequencing and look for base editing

In [61]:
def target_pileup(guide,targets,target_max_diff):
    alt_base = [0] * len(guide)
    ref_base = [0] * len(guide)
    
    for target in targets:
        target_sliced = target[3:23]
        differences = [0 if (x == y) else 1 for (x,y) in zip(guide,target_sliced)]
        prop_diff = float(sum(differences))/float(len(guide))
        
        if prop_diff < target_max_diff:
            for index,diff in enumerate(differences):
                if diff == 0:
                    ref_base[index] += 1
                elif diff == 1:
                    alt_base[index] += 1
                else:
                    print("It's also bad if we end up here")
    
    count = alt_base[0] + ref_base[0]
    editing = [float(a)/float(a+r+0.0000001) for (a,r) in zip(alt_base,ref_base)]
    return((count,editing))

target_pileup("GTATAGGTGATCACCTATAC",["CCGGTATAGGTGGTCACCTATACCGG"],0.2)

(1,
 [0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.9999999000000099,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0])

#### now output the base-editing counts per position

In [7]:
pal_count = 0
matching_reads = 0

output = open("palindromes_base_counts.txt","w")
output.write("guide\ttarget_count\t" + "\t".join(["base" + str(x) for x in range(1,21)]) + "\n")

for key,vl in palendromes.items():
    if key in palendromes_to_outcomes:
        pal_count += 1
        matching_reads += len(palendromes_to_outcomes[key])
        count,editing = target_pileup(key,palendromes_to_outcomes[key])
        output.write(key + "\t" + str(count) + "\t" + "\t".join([str(x) for x in editing]) + "\n")
        
print(pal_count)
print(matching_reads)
output.close()

310
337078


#### what proportion of the palindromes have the correct target sequence? how many have another target sequence instead?

In [52]:
output = open("palindrome_sequence_controls.txt","w")
output.write("guide\tsequence_count\tpercent_correct_target\tpercent_other_target\tnumber_of_other_targets\n")
for key in palendromes.keys():
    if key in palendromes_to_outcomes and len(palendromes_to_outcomes[key]) > 0:
        alt_target_counts = {}
        for x in palendromes_to_outcomes[key]:
            if x[3:23] in palendromes:
                alt_target_counts[x[3:23]] = alt_target_counts.get(x[3:23],0) + 1
        
        counts = [1 if x[3:23] == key else 0 for x in palendromes_to_outcomes[key]]
        alt_counts = [1 if x[3:23] in palendromes else 0 for x in palendromes_to_outcomes[key]]
        output.write(key + "\t" + str(len(palendromes_to_outcomes[key])) + "\t" + str(100.0 * sum(counts)/len(palendromes_to_outcomes[key])) + "\t" + str(100.0 * sum(alt_counts)/len(palendromes_to_outcomes[key])) + "\t" + str(len(alt_target_counts)) + "\n")
output.close()

In [45]:
total_cnt

310

### Now load up the edited library and try to determine 

In [55]:
edited_patterns = "../../2019_11_27_Maryam_palendromes/aligned_sequences.fasta.gz"

In [56]:
ref = ""
sequence = ""
ref_name_set = False
seq_name_set = False

palendromes_to_outcomes_edited = defaultdict(list)

with gzip.open(edited_patterns, 'rt') as f:
    for line in f:
        if line.startswith('>'):
            if not ref_name_set:
                ref_name_set = True
            elif (not seq_name_set) and ref_name_set:
                seq_name_set = True
            elif seq_name_set and ref_name_set:
                (guide,target) = process_ref_seq_alignment(ref,sequence)
                if sum([1 if x == '-' else 0 for x in sequence]) < 30:
                    palendromes_to_outcomes_edited[guide].append(target)
                ref = ""
                sequence = ""
                ref_name_set = True
                seq_name_set = False
            else:
                print("WE SHOULDNT BE HERE")
        else:
            if ref_name_set and (not seq_name_set):
                ref += line.strip("\n")
            else:
                sequence += line.strip("\n")

In [57]:
output = open("palindrome_sequence_cases.txt","w")
output.write("guide\tsequence_count\tpercent_correct_target\tpercent_other_target\tnumber_of_other_targets\n")
for key in palendromes.keys():
    if key in palendromes_to_outcomes_edited and len(palendromes_to_outcomes[key]) > 0:
        alt_target_counts = {}
        for x in palendromes_to_outcomes_edited[key]:
            if x[3:23] in palendromes:
                alt_target_counts[x[3:23]] = alt_target_counts.get(x[3:23],0) + 1
        
        counts = [1 if x[3:23] == key else 0 for x in palendromes_to_outcomes_edited[key]]
        alt_counts = [1 if x[3:23] in palendromes else 0 for x in palendromes_to_outcomes[key]]
        output.write(key + "\t" + str(len(palendromes_to_outcomes_edited[key])) + "\t" + str(100.0 * sum(counts)/len(palendromes_to_outcomes_edited[key])) + "\t" + str(100.0 * sum(alt_counts)/len(palendromes_to_outcomes_edited[key])) + "\t" + str(len(alt_target_counts)) + "\n")
output.close()

In [68]:
pal_count = 0
matching_reads = 0
cutoff = 0.05
processed_reads = 0
output = open("palindromes_base_counts_edited.txt","w")
output.write("guide\ttarget_count\t" + "\t".join([str(x) for x in range(1,21)]) + "\n")
for key,vl in palendromes.items():
    if key in palendromes_to_outcomes_edited:
        pal_count += 1
        matching_reads += len(palendromes_to_outcomes_edited[key])
        count,editing = target_pileup(key,palendromes_to_outcomes_edited[key], cutoff)
        output.write(key + "\t" + str(count) + "\t" + "\t".join([str(x) for x in editing]) + "\n")
output.close()

output = open("palindromes_base_counts_control.txt","w")
output.write("guide\ttarget_count\t" + "\t".join([str(x) for x in range(1,21)]) + "\n")
for key,vl in palendromes.items():
    if key in palendromes_to_outcomes:
        pal_count += 1
        matching_reads += len(palendromes_to_outcomes[key])
        count,editing = target_pileup(key,palendromes_to_outcomes[key], cutoff)
        processed_reads += count
        output.write(key + "\t" + str(count) + "\t" + "\t".join([str(x) for x in editing]) + "\n")
output.close()

print(pal_count)
print(processed_reads)


619
44636
